# 📊 Exploratory Data Analysis: Consolidated 4-Class Dataset & Domain Confound Analysis

> **Classes:** `Normal`, `Bacterial Pneumonia`, `Viral Pneumonia`, `Tuberculosis`  
> **Manifest File:** `data/processed/manifest.csv`  
> **Split Directory:** `data/processed/splits/`

---

## 🎯 Purpose & Research Context

1. **Integrated 4-Class Pipeline:** Combines Kermany (pediatric pneumonia) + Shenzhen (adult TB) + Montgomery (adult held-out external test).
2. **Domain Confound Analysis (Mandatory Thesis Requirement):**
   - **Pediatric (Kermany):** Normal, Bacterial, Viral classes.
   - **Adult (Shenzhen & Montgomery):** Tuberculosis class.
   - **Confound Risk:** The model could potentially learn "child vs adult anatomy" rather than "pneumonia vs tuberculosis pathology".
   - **Mitigation:** Transparent disclosure in paper/defense + benchmark clean 3-class Kermany model alongside 4-class demo model + external test validation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

MANIFEST_PATH = Path("../data/processed/manifest.csv")
SPLIT_DIR = Path("../data/processed/splits")

df_manifest = pd.read_csv(MANIFEST_PATH)
df_train = pd.read_csv(SPLIT_DIR / "train.csv")
df_val = pd.read_csv(SPLIT_DIR / "val.csv")
df_test = pd.read_csv(SPLIT_DIR / "test.csv")
df_ext = pd.read_csv(SPLIT_DIR / "external_test.csv")

print(f"Manifest total unique images: {len(df_manifest)}")


## 1. Consolidated Manifest Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By Label
label_counts = df_manifest['label'].value_counts()
sns.barplot(x=label_counts.index, y=label_counts.values, ax=axes[0], palette="Blues_r")
axes[0].set_title("Overall Manifest Image Count by Diagnosis Label", fontsize=13, fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)
for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height() + 30), ha='center', fontweight='bold')

# By Source
source_counts = df_manifest['source'].value_counts()
sns.barplot(x=source_counts.index, y=source_counts.values, ax=axes[1], palette="Dark2")
axes[1].set_title("Image Count by Dataset Source", fontsize=13, fontweight='bold')
for p in axes[1].patches:
    axes[1].annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height() + 30), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()


## 2. Train / Val / Test / External Split Proportions

In [ ]:
split_data = []
for name, d in [("Train", df_train), ("Val", df_val), ("Test (Internal)", df_test), ("External Test (Montgomery)", df_ext)]:
    for label, count in d['label'].value_counts().items():
        split_data.append({"Split": name, "Label": label, "Count": count})

df_splits = pd.DataFrame(split_data)

plt.figure(figsize=(12, 6))
sns.barplot(data=df_splits, x="Split", y="Count", hue="Label", palette="Set2")
plt.title("Class Distribution Across Patient-Level Data Splits", fontsize=14, fontweight='bold')
plt.ylabel("Number of Images")
plt.legend(title="Diagnosis")
plt.show()


## 3. Pediatric vs Adult Domain Confound Visual Comparison

We plot side-by-side pediatric X-ray (Kermany) vs adult X-ray (Shenzhen/Montgomery) to visualize anatomical differences.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

pediatric_sample = df_manifest[df_manifest['domain'] == 'pediatric'].sample(1, random_state=42).iloc[0]
adult_sample = df_manifest[df_manifest['domain'] == 'adult'].sample(1, random_state=42).iloc[0]

img_ped = Image.open(pediatric_sample['filepath']).convert('L')
img_adult = Image.open(adult_sample['filepath']).convert('L')

axes[0].imshow(img_ped, cmap='gray')
axes[0].set_title(f"PEDIATRIC DOMAIN ({pediatric_sample['source'].upper()})
Label: {pediatric_sample['label']}", fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(img_adult, cmap='gray')
axes[1].set_title(f"ADULT DOMAIN ({adult_sample['source'].upper()})
Label: {adult_sample['label']}", fontsize=12, fontweight='bold')
axes[1].axis('off')

plt.suptitle("Domain Confound Visualization: Pediatric vs Adult Chest Geometry", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


## 📌 Thesis Defense & Research Paper Strategy

| Problem | Scientific Mitigation | Presentation in Thesis / Paper |
| :--- | :--- | :--- |
| **Domain Confound (Pediatric vs Adult)** | Run 3-class clean benchmark (Kermany only) in parallel with 4-class model. | Disclosed explicitly in Limitations section. Highlighted as reason for external validation. |
| **Class Imbalance** | Apply `WeightedRandomSampler` during PyTorch training. | Reported via macro F1-score & AUC (not just overall accuracy). |
| **External Generalization** | Validate on 100% held-out Montgomery source. | Honest drop in performance reported & discussed. |
